# Segmenting a Global Climate Dataset into Per-country `pandas.DataFrames`

For this example notebook of the `globaldatatoolkit.dataset` subpackage's `bycountry` module, 
we will be using a dataset from OpenML which contains climate records from different 
countries spanning 25 years.

> The dataset is an unofficial dataset posted on OpenML called
> [Temperature Emissions Environmental Trends 2000-2024](https://www.openml.org/search?type=data&sort=qualities.NumberOfInstances&status=active&id=46731).

## Importing the Data

Start by importing the data into the runtime. For that, we require Pandas for
its DataFrame object type.

In [1]:
import pandas as pd

dataset_df = pd.read_csv("data/climate_2000_2024.csv")

In [2]:
dataset_df.head()

,Year,Country,Avg Temperature (°C),CO2 Emissions (Tons/Capita),Sea Level Rise (mm),Rainfall (mm),Population,Renewable Energy (%),Extreme Weather Events,Forest Area (%)
0,2006,UK,8.9,9.3,3.1,1441,530911230,20.4,14,59.8
1,2019,USA,31.0,4.8,4.2,2407,107364344,49.2,8,31.0
2,2014,France,33.9,2.8,2.2,1241,441101758,33.3,9,35.5
3,2010,Argentina,5.9,1.8,3.2,1892,1069669579,23.7,7,17.7
4,2007,Germany,26.9,5.6,2.4,1743,124079175,12.5,4,17.4


Now, we will use the `global-data-toolkit` package to load data into
per-country DataFrames.

First, we import the package and initalize the DataLoader, which generates the
csv files.

In [3]:
from globaldatatoolkit.dataloaders import bycountry

per_country_data_loader = bycountry.DataLoader(
    "data/climate_2000_2024.csv", "data"
)

Next, we will load the data into runtime memory as a map of country names to
their climate data DataFrames.

In [4]:
country_name_to_df_map = per_country_data_loader.load()

In [5]:
country_name_to_df_map["Argentina"].head()

,Year,Avg Temperature (°C),CO2 Emissions (Tons/Capita),Sea Level Rise (mm),Rainfall (mm),Population,Renewable Energy (%),Extreme Weather Events,Forest Area (%)
0,2010,5.9,1.8,3.2,1892,1069669579,23.7,7,17.7
1,2006,30.7,11.6,3.9,1755,586706107,41.9,10,50.5
2,2004,9.8,2.1,1.5,1290,742981303,15.6,1,40.1
3,2008,18.0,7.1,4.8,1037,1209391736,27.8,4,47.1
4,2013,25.4,9.2,3.7,2881,1075399963,43.9,14,30.9


Before data can be visualized or analyzed, we study the properties available:

In [6]:
country_name_to_df_map["Argentina"].columns

Index(['Year', 'Avg Temperature (°C)', 'CO2 Emissions (Tons/Capita)',
       'Sea Level Rise (mm)', 'Rainfall (mm)', 'Population',
       'Renewable Energy (%)', 'Extreme Weather Events', 'Forest Area (%)'],
      dtype='object')

### Data Properties

Upon printing the columns, the following properties are available for each data
point:

- year
- avg temperature (°C)
- CO2 emissions (tons/capita)
- sea level rise (mm)
- rainfall (mm)
- population
- renewable energy (%)
- extreme weather events
- forest area (%)

Now let's gauge the volume of data available.

In [7]:
for country_name, country_df in country_name_to_df_map.items():
  print(country_name + ": " + str(len(country_df)))

UK: 65
Japan: 63
Argentina: 67
India: 70
China: 67
Russia: 74
Indonesia: 75
Mexico: 55
USA: 73
France: 66
Brazil: 67
Germany: 61
South Africa: 73
Canada: 67
Australia: 57


### Observation: Number of Years < Number of Records for each country

It seems like most countries' data points fall between the 50 and 70 quantities.

We have 25 year values available and multiple climate records of a country for
the same year. Since we have no information about what is different between such
records (which have the same year and country), we will have to compare some of
these instances ourselves.

In [8]:
# dictionaries are passed by reference
def sort_dfs_by_year(name_to_df_map: dict[str, pd.DataFrame]):
  """sort the pandas.DataFrame objects in the provided dictionary by year"""
  for country_name, country_df in name_to_df_map.items():
    name_to_df_map[country_name] = country_df.sort_values(by=["Year"])

sort_dfs_by_year(country_name_to_df_map)

country_name_to_df_map["Russia"].head()


,Year,Avg Temperature (°C),CO2 Emissions (Tons/Capita),Sea Level Rise (mm),Rainfall (mm),Population,Renewable Energy (%),Extreme Weather Events,Forest Area (%)
54,2000,6.9,4.0,4.8,560,1051785465,32.3,5,26.9
61,2000,15.1,15.8,5.0,1138,242436875,37.6,1,52.4
40,2001,33.9,12.6,4.8,1707,877308425,6.7,9,21.0
20,2001,19.4,3.0,3.0,839,1035525610,49.1,7,66.5
0,2002,26.4,6.9,2.1,2393,29715260,36.9,1,53.0


It is observed upon comparing different records of the same year that values of properties such as average temperature, CO2 emissions and even population are vastly different.
It is likely that the datapoints correspond to different regions within a country. No meaningful climate analysis can be conducted if variables such as  location of measurement are unknown or unreliable.
Another dataset with precise location and time measurements must be used for effective analysis.

Datasets such as Berkeley Earth's [National and Regional Temperature Data](https://berkeleyearth.org/data/) may be leveraged.